##### Simple Libraries

In [ ]:
import os
import numpy as np # linear algebra
import pandas as pd 

In [ ]:
from pydantic import BaseModel, Field
from typing import List, Optional, Literal
from typing import TypedDict, Annotated, Optional, List, Literal
from pydantic import BaseModel, EmailStr, Field

##### LangChain Core and Components

In [ ]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import PydanticOutputParser, StrOutputParser, JsonOutputParser
from langchain_core.runnables import RunnableParallel

LangChain Models

In [ ]:
from langchain_anthropic import ChatAnthropic
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_huggingface import ChatHuggingFace, HuggingFacePipeline

###### API Keys

In [ ]:
os.environ["GOOGLE_API_KEY"] = ""
os.environ["ANTHROPIC_API_KEY"] = ""

#### Models

In [ ]:
Amodel = ChatAnthropic(
    model='claude-sonnet-4-5-20250929',
    anthropic_api_key="")
model = ChatGoogleGenerativeAI(model="gemini-2.5-flash")

In [ ]:
while True:
    result = Amodel.invoke('Are you there?')
    print(result.content)
    user_input = input('You: ')
    if(user_input == "Exit"):
        break
    result = Amodel.invoke(user_input)
    print(result.content)

##### Structured Output Parsing

In [ ]:
class Person(TypedDict):
    Name: str
    age: int
    
s_model = Amodel.with_structured_output(Person) 

while True:
    user_input = input('You: ')
    if(user_input == "Exit"):
        break
    result = s_model.invoke(user_input)
    print(result)
    print(result['age'])

In [ ]:
class Person(TypedDict):
    name: Annotated[str, "Complete name"]
    age: Annotated[int, "Age of the person"]
    skills: Annotated[Optional[List[str]], "Skills of the person"]

s_model = model.with_structured_output(Person)

while True:
    user_input = input("You: ")
    if user_input.lower() == "exit":
        break

    result = s_model.invoke(user_input)
    print(result)

In [ ]:
text="""Rahim is a final year CS student. He is 25 years old. 
You can contact him at rahim.cs@fast.edu.pk and rahim.work@gmail.com. 
His final year CGPA is 3.12. """

In [ ]:

class Info(BaseModel):
    Name: str
    Age: int
    Email: list[EmailStr]
    cgpa: float = Field(gt=0.0, lt=4.01, default=2.00, description="This decimal values reprenting the Person marks in final year degree")
s_model = Amodel.with_structured_output(Info)   

while True:
    user_input = text
    if(user_input == "Exit"):
        break
    result = s_model.invoke(user_input)
    print(result)

#### Prompt Template & Chaining With StringOutput Parser


In [ ]:
prompt1 = PromptTemplate(
    template="""
        You are a licensed medical doctor specialized in {specs}.
        
        Your task:
        - Carefully analyze the patient's query.
        - Provide a possible diagnosis based only on the given information.
        - If the information is insufficient or unclear, respond with: "I don't know based on the provided information."
        
        Guidelines:
        - Do NOT fabricate medical facts.
        - Do NOT provide prescriptions or dosage.
        - Keep the explanation clear and professional.
        - Mention possible causes (if applicable).
        - Suggest whether the patient should consult a doctor in person.
        
        Patient Query:
        {query}
        
        Response Format:
        1. Possible Diagnosis:
        2. Reasoning (2-3 bullet points):
        3. Recommendation:
        """,
    input_variables=['specs', 'query']
)


In [ ]:
prompt2 = PromptTemplate(
    template="""
You are a licensed medical doctor.

Based ONLY on the diagnosis information provided below, generate a prescription.

IMPORTANT RULES:
- Return ONLY valid JSON.
- Do NOT include explanations or markdown.

Diagnosis Information:
{input}

Return output in this EXACT JSON format:

{{
  "medicine": "Name of medicine",
  "prescription": "How to take it (dosage instructions)",
  "duration": "How long to take it"
}}
""",
    input_variables=['input']
)

In [ ]:
strparser = StrOutputParser()

In [ ]:
jsonparser = JsonOutputParser()

In [ ]:
chain = prompt1 | Amodel | strparser

In [ ]:
chain.invoke({
    'specs': 'General Physician',
    'query': 'I have had fever (101°F), body aches, mild headache, and sore throat for the last 2 days.'
})

In [ ]:
chain = prompt1 | Amodel | strparser| prompt2 | Amodel | jsonparser

In [ ]:
diagnostician = chain.invoke({
    'specs': 'General Physician',
    'query': 'I have had fever (101°F), body aches, mild headache, and sore throat for the last 2 days.'
})

In [ ]:
diagnostician

In [ ]:
# multiply = lambda x, y: x * y
# print(multiply(3, 4))
# # Use a lambda or a dictionary to map the string to the 'input' key
# chain = (
#     prompt1 
#     | model 
#     | strparser 
#     | {"input": lambda x: x}  # This converts the string to a dict
#     | prompt2 
#     | model 
#     | jsonparser
# )

#### Simple Workflow


In [ ]:
from langchain_core.runnables import RunnableLambda

# 1. Define the components
diagnostician = prompt1 | Amodel | strparser

# Use RunnableLambda to ensure the dictionary is treated as a 'Runnable'
pharmacist = (
    RunnableLambda(lambda x: {"input": x}) 
    | prompt2 
    | Amodel 
    | jsonparser
)

def medical_agent(specs, query):
    print(f"--- Agent: Consulting with {specs} ---")
    
    # STEP 1: The Reasoning Phase
    # Now diagnostician is a proper chain, it has .invoke()
    diagnosis = diagnostician.invoke({'specs': specs, 'query': query})
    print(f"Doctor's Note: {diagnosis}")

    # STEP 2: The Logic Gate
    if "I don't know" in diagnosis:
        return {"status": "Incomplete", "message": "More info needed."}

    # STEP 3: The Action Phase
    print("--- Agent: Generating Prescription JSON ---")
    prescription = pharmacist.invoke(diagnosis)
    
    return {
        "status": "Success",
        "diagnosis": diagnosis,
        "prescription": prescription
    }

# Run it!
result = medical_agent(specs='General Physician', query='Fever 101F, sore throat.')
print(result)

In [ ]:
parser = JsonOutputParser()

In [ ]:
template = PromptTemplate(
    template="""
        You are a clinical information extraction system.
        
        From the patient case below, extract structured medical information.
        
        Required Fields:
        - patient_name
        - age
        - gender
        - symptoms
        - suspected_condition
        - required_specialist
        
        Rules:
        - Do NOT invent missing details.
        - If information is not present, return null.
        - Keep symptom descriptions concise.
        
        Patient Case:
        {input}
        
        Return output strictly in the required structured format:
        {format_instruction}
        """,
    input_variables=["input"],
    partial_variables={
        "format_instruction": parser.get_format_instructions()
    }
)

In [ ]:
parser.get_format_instructions()

In [ ]:
prompt = template.format(
    input="""
Patient Name: Ali Khan
Age: 45
Gender: Male
Symptoms: Persistent chest pain, shortness of breath, sweating for 1 hour.
Doctor suspects possible cardiac issue and recommends cardiology consultation.
"""
)
print(prompt)

In [ ]:
result = Amodel.invoke(prompt)
result

In [ ]:
result.dict()

In [ ]:
parser.parse(result.content)

In [ ]:
chain = template | model | parser 

In [ ]:
text="""
Patient Name: Ali Khan
Age: 45
Gender: Male
Symptoms: Persistent chest pain, shortness of breath, sweating for 1 hour.
Doctor suspects possible cardiac issue and recommends cardiology consultation.
"""

In [ ]:
chain.invoke({text})

#### Example 2: Chaining with Pydantic Parser

In [ ]:
class MCQ(BaseModel):
    question: str = Field(description="The MCQ question")
    options: List[str] = Field(description="List of answer options")
    correct_answer: str = Field(description="Correct option from the list")


class ShortQuestion(BaseModel):
    question: str = Field(description="Short answer question")
    correct_answer: str = Field(description="Correct short answer")


class ExamQuestion(BaseModel):
    type: Literal["MCQ", "SHORT"]
    mcq: Optional[MCQ] = None
    short: Optional[ShortQuestion] = None

In [ ]:
class Exam(BaseModel):
    questions: List[ExamQuestion]


In [ ]:
Pyparser = PydanticOutputParser(pydantic_object=Exam)

In [ ]:
strparser

In [ ]:
prompt1 = PromptTemplate(
    template="""
You are an expert exam compiler.

Generate exactly {number} {type} questions on the topic: {topic}.

Requirements:
- Questions must be clear, precise, and exam-ready.
- Stay strictly within the given topic.
- Avoid repetition.
- Do not include explanations unless required by the format.

Return output strictly according to the schema below:


{format_instruction}
""".strip(),
    input_variables=['number', 'type', 'topic'],
    partial_variables={'format_instruction': Pyparser.get_format_instructions()},
)



In [ ]:
prompt2 = PromptTemplate(
    template="""
You are an exam formatter.

The structured questions provided below were generated in the previous step.

Your task:
- Process the given structured data exactly as received.
- Ensure formatting is clean and consistent.
- Do not modify the content of the questions.
- Do not add explanations or extra text.

Input:
{input}

Return output strictly in the required JSON format.
""".strip(),
    input_variables=['input']
)


In [ ]:
prompt = PromptTemplate(
    template="""
        You are a professional exam paper compiler.
        
        Your task is to generate {number} {type} question(s) on the topic: "{topic}".
        
        Guidelines:
        - Questions must be clear, concise, and academically sound.
        - Ensure alignment with the topic.
        - Avoid ambiguity.
        - Do NOT repeat questions.
        - Match the cognitive level appropriate for university students.
        - If the topic is too vague, return an empty list.
        
        Additional Instructions:
        - Include a short description or context if needed.
        - Ensure each question tests conceptual understanding, not just memorization.
        
        Output Requirements:
        {format_instruction}
        """,
    input_variables=['number', 'type', 'topic'],
    partial_variables={
        'format_instruction': Pyparser.get_format_instructions()
    }
)

In [ ]:
# model = s_model = model.with_structured_output(Exam)

In [ ]:
prompt_u = prompt1.format(
    number=5,
    type="MCQ",
    topic="Python functions"
)

In [ ]:
prompt_u

In [ ]:
s_model = Amodel.with_structured_output(Exam)

In [ ]:
exam_result = s_model.invoke(prompt_u)

In [ ]:
exam_result.dict()

In [ ]:
chain = prompt | Amodel | Pyparser

In [ ]:
result = chain.invoke({
    "number": 3,
    "type": "MCQ",
    "topic": "Python lists"
})
result.dict()

In [ ]:
result.questions[1].dict()


In [ ]:
result.dict()['questions'][0]['mcq']['options']

#### Example 3: Model Tiering & Chaining

###### Chota Model


In [ ]:
llm = HuggingFacePipeline.from_model_id('gokaygokay/prompt-enhancer-gemma-3-270m-it', task='text-generation')
chota_model = ChatHuggingFace(llm = llm)

###### Prompt and Scenario 

In [ ]:
from pydantic import BaseModel, Field
from typing import List

class Critique(BaseModel):
    motivation: str = Field(
        description="Summarize the primary research motivation and problem the paper aims to address."
    )
    
    strengths: List[str] = Field(
        description="List the major contributions, innovations, or strengths of the proposed work."
    )
    
    weaknesses: List[str] = Field(
        description="Identify the main limitations, assumptions, or weaknesses of the proposed approach."
    )
    
    future_directions: List[str] = Field(
        description="Suggest potential future research directions or possible improvements."
    )


In [ ]:
strparser = StrOutputParser()

In [ ]:
pyparser = PydanticOutputParser(pydantic_object=Critique)

In [ ]:
prompt = PromptTemplate(
    template = """You are an academic researcher. 

Provide a structured and critical evaluation of the following research study: {text}

Your critique should clearly analyze:
- The research motivation
- The key strengths and major contributions
- The main limitations or weaknesses
- Potential future research directions

Return the response strictly according to the specified format below:
{format_instruction}""",
    input_variables = ['text'],
    partial_variables = {'format_instruction': pyparser.get_format_instructions()}
)


###### Parser


In [ ]:
prompt.invoke('Attention is all you need')

###### Chaining with larger Model of Gemnie and Small model of Gemma

In [ ]:
# prompt = prompt.invoke('Attention is All you Need')

In [ ]:
# chota_model.invoke(prompt.invoke('Attention is All you Need'))

In [ ]:
chain = prompt | chota_model 

In [ ]:
chain.invoke('Attention is All you Need by google brain').model_dump()

In [ ]:
chain = prompt | chota_model | strparser 

In [ ]:
chain.invoke('Attention is All you Need by google brain')

In [ ]:
chain = prompt | chota_model | strparser | Amodel | pyparser


In [ ]:
chain.invoke('Attention is All you Need by google brain').model_dump()

In [ ]:
chain.get_graph().print_ascii()

In [ ]:
chain.invoke('Attention is All you Need by google brain').model_dump()

##### Parallel Chains

In [ ]:
prompt_q = PromptTemplate(
    template = "Genrate Five technical question from the research paper {text}",
    input_variable = ['text']
)

prompt_s = PromptTemplate(
    template = "Genrate Five research question from the research paper {text}",
    input_variable = ['text']
)

prompt_m = PromptTemplate(
    template = "Merger the questions and rephrased the look a like questions from both sorce given below.\n sorce 1 : {tech}, source2 : {research}",
    input_variable = ['tech','research']
)


In [ ]:
parser = StrOutputParser()

In [ ]:
parallel_chain = RunnableParallel({
    'tech' : prompt_q | Amodel | parser,
    'research' : prompt_s | Amodel | parser
})

In [ ]:
combined = prompt_m | Amodel | parser

In [ ]:

chain = parallel_chain | combined

In [ ]:
chain.get_graph().print_ascii()

In [ ]:
chain.invoke('Attention is all you Need')


In [ ]:
chain.get_graph().print_ascii()


In [ ]:
chain.invoke('Attention is All you Need by google brain')

In [ ]:
template_text = "Your text here for Notes Genration"

In [ ]:
class ContentBlock(BaseModel):
    subsection_title: str = Field(..., description="The title of the subsection")
    explanation: str = Field(..., description="Deep dive explanation of the concept")
    text_examples: Optional[List[str]] = Field(None, description="Prose-based examples or use cases")
    code_snippets: Optional[List[str]] = Field(None, description="Executable Python or LangChain code")
    images: Optional[List[str]] = Field(None, description="Images related to given section")

class Section(BaseModel):
    section_title: str = Field(..., description="Main heading (e.g., Agent Components, Prompt Engineering)")
    subsections: List[ContentBlock]

class TechnicalNotes(BaseModel):
    document_title: str
    content: List[Section]

In [ ]:
pyparser = PydanticOutputParser(pydantic_object=TechnicalNotes)

In [ ]:
prompt = PromptTemplate(
    template=template_text,  # Use the variable that holds the """string"""
    input_variables=[],
    partial_variables={'format_instructions': pyparser.get_format_instructions()}
)

In [ ]:
chain = prompt | Amodel | pyparser

In [ ]:
result = chain.invoke({})

In [ ]:
result